# Phase 4: ARMA Model Identification

**Objective:** Implement systematic ARMA order identification via AICC grid search on three Phase 3 approach residuals (classical decomposition, differencing ∇₇, harmonic GLS). Verify causality/invertibility constraints per B&D §3.1, confirm model adequacy via Ljung-Box whiteness test (B&D §9.1), and output per-approach AICC winners.

**Reference:** Brockwell & Davis, *Introduction to Time Series and Forecasting*, 3rd ed., §3.1 (causality/invertibility), §5.2 (MLE), §5.5.2 (AICC), §1.6(b) (Ljung-Box test)

## Section 1: Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import statsmodels for ARIMA and diagnostics
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Import project configuration
import config

In [ ]:
# Verify key config constants
print(f"AT_LOAD_DAILY_PATH: {config.AT_LOAD_DAILY_PATH}")
print(f"DECOMPOSITION_PATH: {config.DECOMPOSITION_PATH}")
print(f"MODEL_SELECTION_PATH: {config.MODEL_SELECTION_PATH}")
print(f"MAX_P: {config.MAX_P}, MAX_Q: {config.MAX_Q}")
print(f"FIGURE_DPI: {config.FIGURE_DPI}")

# Ensure results directory exists
Path(config.RESULTS_DIR).mkdir(exist_ok=True)
print(f"Results directory ready: {config.RESULTS_DIR}")

## Section 2: Load Training Data and Phase 3 Coefficients

In [ ]:
# Load training data from Phase 1 (daily load, 2015-2024)
daily_df = pd.read_csv(config.AT_LOAD_DAILY_PATH, parse_dates=['date'])
daily_train = daily_df[daily_df['date'] <= pd.Timestamp(config.TRAIN_END)].reset_index(drop=True)
X_t = daily_train['load_mw'].values
n_obs_original = len(X_t)

print(f"Training window: {daily_train['date'].min().date()} to {daily_train['date'].max().date()}")
print(f"Total training observations: {n_obs_original}")
assert n_obs_original == 3653, f"Expected 3653 training rows, got {n_obs_original}"

In [ ]:
# Load Phase 3 decomposition.json for approach coefficients
with open(config.DECOMPOSITION_PATH) as f:
    decomp = json.load(f)

print("Phase 3 decomposition loaded.")
print(f"Available approaches in phase3 section: {list(decomp.get('phase3', {}).keys())}")

## Section 2.1: Classical Decomposition Residuals (Harmonic Regression OLS)

**Method:** Harmonic regression with polynomial trend per B&D §1.5.1(d).

Formula: R_t = X_t − m̂_t(poly) − ŝ_t(weekly) − ŝ_t(annual)

where design matrix includes:
- Constant and polynomial trend terms (order 2)
- Harmonic pairs at frequencies 1/7, 2/7, ..., K₁/7 (weekly)
- Harmonic pairs at frequencies 1/365.25, 2/365.25, ..., K₂/365.25 (annual)

In [ ]:
# Extract classical approach parameters from Phase 3 or Phase 2 (fallback)
K1_classical = 3  # Weekly harmonic pairs
K2_classical = 21  # Annual harmonic pairs
trend_order_classical = 2  # Quadratic trend

if 'phase3' in decomp and 'classical' in decomp['phase3']:
    print("Using Phase 3 classical metadata...")
    phase3_classical = decomp['phase3']['classical']
    # Phase 3 may not explicitly store K1, K2; use Phase 2 strategy_a

if 'strategies' in decomp and 'strategy_a' in decomp['strategies']:
    strategy_a = decomp['strategies']['strategy_a']
    K1_classical = int(strategy_a.get('K1', 3))
    K2_classical = int(strategy_a.get('K2_selected', 21))
    trend_order_classical = int(strategy_a.get('trend_order_selected', 2))

print(f"Classical approach: K1={K1_classical}, K2={K2_classical}, trend_order={trend_order_classical}")

In [ ]:
# Build harmonic design matrix for classical decomposition
t = np.arange(n_obs_original)

# Start with constant and polynomial terms
cols = [np.ones(n_obs_original)]

# Add weekly harmonic pairs
for k in range(1, K1_classical + 1):
    cols.append(np.sin(2 * np.pi * k * t / 7))
    cols.append(np.cos(2 * np.pi * k * t / 7))

# Add annual harmonic pairs
for k in range(1, K2_classical + 1):
    cols.append(np.sin(2 * np.pi * k * t / 365.25))
    cols.append(np.cos(2 * np.pi * k * t / 365.25))

# Add polynomial trend terms
cols.append(t)
for p in range(2, trend_order_classical + 1):
    cols.append(t ** p)

X_classical = np.column_stack(cols)
print(f"Classical design matrix shape: {X_classical.shape}")
print(f"Expected: ({n_obs_original}, {1 + 2*K1_classical + 2*K2_classical + trend_order_classical})")

In [ ]:
# Fit OLS: β = (X'X)^-1 X'y (B&D §1.5.1(d))
beta_classical, _, _, _ = np.linalg.lstsq(X_classical, X_t, rcond=None)
print(f"OLS solution: {len(beta_classical)} coefficients estimated")

# Compute residuals
residual_classical_vals = X_t - X_classical @ beta_classical
residual_classical = pd.Series(
    residual_classical_vals,
    index=daily_train['date'],
    name='residual_classical'
)

print(f"Classical residuals: n={len(residual_classical)}, mean={residual_classical.mean():.2f}, std={residual_classical.std():.2f}")
print(f"ACF(1): {residual_classical.autocorr(1):.4f}")

## Section 2.2: Differencing Residuals (∇₇ Operator)

**Method:** Seasonal differencing at lag 7 per B&D §6.4.

Formula: Y_t = X_t − X_{t−7} (∇₇ operator)

Applies to training series; first 7 observations lost.

In [ ]:
# Apply ∇₇ differencing (B&D §6.5)
residual_differencing_vals = np.diff(X_t, n=1) if False else (X_t[7:] - X_t[:-7])  # Equivalent to diff(7)
residual_differencing = pd.Series(
    residual_differencing_vals,
    index=daily_train['date'].iloc[7:].reset_index(drop=True),
    name='residual_differencing'
)

print(f"Differencing residuals: n={len(residual_differencing)}, mean={residual_differencing.mean():.2f}, std={residual_differencing.std():.2f}")
print(f"ACF(1): {residual_differencing.autocorr(1):.4f}")
assert len(residual_differencing) == 3646, f"Expected 3646 residuals after ∇₇, got {len(residual_differencing)}"

## Section 2.3: Harmonic GLS Residuals

**Method:** Generalized least squares (GLS) with harmonic basis per B&D §6.6 and Phase 3 implementation.

Formula: R_t = X_t − β̂_GLS^T [design matrix]

where design matrix has K₁=3 weekly harmonics and K₂=10 annual harmonics (per Phase 3 GLS settings).

In [ ]:
# Extract harmonic GLS parameters and coefficients from Phase 3
K1_gls = 3
K2_gls = 10
beta_gls_dict = None

if 'phase3' in decomp and 'harmonic_gls' in decomp['phase3']:
    hg = decomp['phase3']['harmonic_gls']
    K2_gls = int(hg.get('k2_selected', 10))
    if 'beta_coefficients' in hg:
        beta_gls_dict = hg['beta_coefficients']
        print(f"Harmonic GLS: K1={K1_gls}, K2={K2_gls}, {len(beta_gls_dict)} beta coefficients loaded")

if beta_gls_dict is None:
    print(f"No GLS beta coefficients found; will fall back to OLS with K1={K1_gls}, K2={K2_gls}")

In [ ]:
# Build harmonic design matrix for GLS (same structure as classical, different K values)
cols_gls = [np.ones(n_obs_original)]

# Weekly harmonics
for k in range(1, K1_gls + 1):
    cols_gls.append(np.sin(2 * np.pi * k * t / 7))
    cols_gls.append(np.cos(2 * np.pi * k * t / 7))

# Annual harmonics
for k in range(1, K2_gls + 1):
    cols_gls.append(np.sin(2 * np.pi * k * t / 365.25))
    cols_gls.append(np.cos(2 * np.pi * k * t / 365.25))

# Polynomial trend (quadratic)
cols_gls.append(t)
cols_gls.append(t ** 2)

X_gls = np.column_stack(cols_gls)
print(f"GLS design matrix shape: {X_gls.shape}")

In [ ]:
# Use stored GLS beta coefficients if available, otherwise fit OLS
if beta_gls_dict is not None:
    # Convert dict {"beta_0", "beta_1", ...} to numpy array
    beta_gls = np.array([beta_gls_dict.get(f"beta_{i}", 0.0) for i in range(X_gls.shape[1])])
    print(f"Using stored GLS coefficients: {len(beta_gls)} values")
else:
    # Fallback: OLS fit
    beta_gls, _, _, _ = np.linalg.lstsq(X_gls, X_t, rcond=None)
    print(f"Using OLS fallback: {len(beta_gls)} coefficients estimated")

# Compute residuals
residual_harmonic_gls_vals = X_t - X_gls @ beta_gls
residual_harmonic_gls = pd.Series(
    residual_harmonic_gls_vals,
    index=daily_train['date'],
    name='residual_harmonic_gls'
)

print(f"Harmonic GLS residuals: n={len(residual_harmonic_gls)}, mean={residual_harmonic_gls.mean():.2f}, std={residual_harmonic_gls.std():.2f}")
print(f"ACF(1): {residual_harmonic_gls.autocorr(1):.4f}")

## Section 3: ACF/PACF Visual Identification

**Reference:** B&D §3.2 — Visual identification of AR and MA orders from ACF and PACF plots.

**Interpretation guidance:**
- ACF cuts off at lag p → MA(p)
- PACF cuts off at lag q → AR(q)
- Both decay slowly → ARMA(p,q) likely

### 3.1 Classical Decomposition Residuals ACF/PACF

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
plot_acf(residual_classical, lags=60, ax=axes[0], title="ACF (B&D §1.4)")
plot_pacf(residual_classical, lags=60, ax=axes[1], title="PACF (B&D §3.2)")
axes[0].set_ylabel('ACF')
axes[1].set_ylabel('PACF')
plt.suptitle('Classical Decomposition Residuals: ACF/PACF (60 lags)', fontsize=13, y=1.00)
plt.tight_layout()
plt.savefig(config.RESULTS_DIR / 'acf_pacf_classical.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

print(f"ACF(1)={residual_classical.autocorr(1):.3f}, ACF(7)={residual_classical.autocorr(7):.3f}, ACF(365)={residual_classical.autocorr(365) if len(residual_classical) > 365 else np.nan:.3f}")

**Observation:** Classical residuals show significant autocorrelation at lag 1 and moderate decay thereafter. PACF suggests potential AR order around 2-3; ACF suggests possible MA order. ARMA(p,q) with p,q ≥ 1 likely.

### 3.2 Differencing Residuals ACF/PACF

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
plot_acf(residual_differencing, lags=60, ax=axes[0], title="ACF (B&D §1.4)")
plot_pacf(residual_differencing, lags=60, ax=axes[1], title="PACF (B&D §3.2)")
axes[0].set_ylabel('ACF')
axes[1].set_ylabel('PACF')
plt.suptitle('Differencing (∇₇) Residuals: ACF/PACF (60 lags)', fontsize=13, y=1.00)
plt.tight_layout()
plt.savefig(config.RESULTS_DIR / 'acf_pacf_differencing.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

print(f"ACF(1)={residual_differencing.autocorr(1):.3f}, ACF(7)={residual_differencing.autocorr(7):.3f}, ACF(365)={residual_differencing.autocorr(365) if len(residual_differencing) > 365 else np.nan:.3f}")

**Observation:** Differencing residuals show moderate positive autocorrelation at lag 1; note seasonal spike at lag 365 (ACF ≈ +0.226 per Phase 3 diagnostics), expected from ∇₇ alone without annual removal. PACF suggests AR component; ARMA(p,q) with p≥1 likely.

### 3.3 Harmonic GLS Residuals ACF/PACF

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
plot_acf(residual_harmonic_gls, lags=60, ax=axes[0], title="ACF (B&D §1.4)")
plot_pacf(residual_harmonic_gls, lags=60, ax=axes[1], title="PACF (B&D §3.2)")
axes[0].set_ylabel('ACF')
axes[1].set_ylabel('PACF')
plt.suptitle('Harmonic GLS Residuals: ACF/PACF (60 lags)', fontsize=13, y=1.00)
plt.tight_layout()
plt.savefig(config.RESULTS_DIR / 'acf_pacf_harmonic_gls.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

print(f"ACF(1)={residual_harmonic_gls.autocorr(1):.3f}, ACF(7)={residual_harmonic_gls.autocorr(7):.3f}, ACF(365)={residual_harmonic_gls.autocorr(365) if len(residual_harmonic_gls) > 365 else np.nan:.3f}")

**Observation:** Harmonic GLS residuals exhibit similar structure to classical (both n=3653, harmonic-based). ACF decays gradually; PACF suggests AR order. Expected similarity to classical approach given design basis overlap.

## Section 4: AICC Grid Search — ARIMA(p,0,q) Model Selection

**Reference:** B&D §5.5.2 (AICC criterion), §5.2 (MLE)

**Method:** Fit all ARIMA(p,0,q) with p,q ∈ {0,…,5} (36 models per approach). Use exact MLE via state-space representation. Skip (0,0,0) (white noise). Store AICC, root moduli, convergence status.

In [ ]:
def fit_arima_grid(residuals, approach_name, max_p=5, max_q=5):
    """
    Fit ARIMA(p,0,q) grid for all (p,q) combinations.
    
    Per B&D §5.2 exact MLE; trend='n' (no constant on residuals).
    Skip (0,0,0) — white noise model.
    
    Args:
        residuals: pd.Series or np.array of residual values
        approach_name: str, for logging
        max_p, max_q: int, grid bounds
        
    Returns:
        pd.DataFrame with columns: p, q, aicc, ar_root_min, ma_root_min, causal, invertible, convergence
    """
    results_list = []
    n_obs = len(residuals)
    
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            if p == 0 and q == 0:
                continue  # Skip white noise model
            
            try:
                # Fit ARIMA(p,0,q) with exact MLE (B&D §5.2)
                model = ARIMA(residuals, order=(p, 0, q), trend='n')
                result = model.fit(method='statespace')
                
                # Extract roots (B&D §3.1 causality/invertibility)
                ar_moduli = np.abs(result.arroots) if len(result.arroots) > 0 else np.array([np.inf])
                ma_moduli = np.abs(result.maroots) if len(result.maroots) > 0 else np.array([np.inf])
                
                causal = bool(np.all(ar_moduli > 1.001))  # D-09: threshold 1.001
                invertible = bool(np.all(ma_moduli > 1.001))
                
                results_list.append({
                    'p': p,
                    'q': q,
                    'aicc': float(result.aicc),  # B&D §5.5.2 eq 5.5.4
                    'ar_root_min': float(np.min(ar_moduli)) if len(ar_moduli) > 0 else np.inf,
                    'ma_root_min': float(np.min(ma_moduli)) if len(ma_moduli) > 0 else np.inf,
                    'causal': causal,
                    'invertible': invertible,
                    'convergence': True,
                })
            except Exception as e:
                # Non-convergence, singular matrix, etc.
                results_list.append({
                    'p': p,
                    'q': q,
                    'aicc': np.nan,
                    'ar_root_min': np.nan,
                    'ma_root_min': np.nan,
                    'causal': False,
                    'invertible': False,
                    'convergence': False,
                })
    
    df = pd.DataFrame(results_list)
    return df.sort_values('aicc').reset_index(drop=True)

print("AICC grid fitting function defined.")

In [ ]:
# Fit grids for all three approaches
print("\nFitting AICC grids...\n")

results_classical_df = fit_arima_grid(residual_classical, "classical", max_p=config.MAX_P, max_q=config.MAX_Q)
print(f"Classical: {len(results_classical_df)} models fitted, {results_classical_df['convergence'].sum()} converged")

results_differencing_df = fit_arima_grid(residual_differencing, "differencing", max_p=config.MAX_P, max_q=config.MAX_Q)
print(f"Differencing: {len(results_differencing_df)} models fitted, {results_differencing_df['convergence'].sum()} converged")

results_harmonic_gls_df = fit_arima_grid(residual_harmonic_gls, "harmonic_gls", max_p=config.MAX_P, max_q=config.MAX_Q)
print(f"Harmonic GLS: {len(results_harmonic_gls_df)} models fitted, {results_harmonic_gls_df['convergence'].sum()} converged")

## Section 5: Causality and Invertibility Verification

**Reference:** B&D §3.1 — All AR and MA roots must lie strictly outside the unit circle.

**Threshold:** Modulus > 1.001 per decision D-09 (conservative margin for numerical stability).

In [ ]:
# Root verification summary per approach
for approach_name, df in [
    ("classical", results_classical_df),
    ("differencing", results_differencing_df),
    ("harmonic_gls", results_harmonic_gls_df),
]:
    causal_count = df['causal'].sum()
    invertible_count = df['invertible'].sum()
    valid_count = (df['causal'] & df['invertible']).sum()
    
    print(f"\n{approach_name.upper()}:")
    print(f"  Total models: {len(df)}")
    print(f"  Causal (AR roots |z| > 1.001): {causal_count}/{len(df)}")
    print(f"  Invertible (MA roots |z| > 1.001): {invertible_count}/{len(df)}")
    print(f"  Causal AND invertible: {valid_count}/{len(df)}")
    
    if valid_count > 0:
        top_valid = df[df['causal'] & df['invertible']].iloc[0]
        print(f"  Top AICC (causal+invertible): ARIMA({int(top_valid['p'])},0,{int(top_valid['q'])}) AICC={top_valid['aicc']:.1f}")
    else:
        print(f"  WARNING: No causal+invertible models found!")

## Section 6: Ljung-Box Whiteness Test on Top-3 Models

**Reference:** B&D §1.6(b), §9.1 — Ljung-Box test statistic for model adequacy.

**Test:** Ljung-Box h=20 on residuals of each top-3 model.

**Criterion:** p-value > 0.05 indicates white noise (adequate model).

In [ ]:
def ljung_box_test_top3(results_df, residuals_series, approach_name):
    """
    Test top-3 causal+invertible models with Ljung-Box h=20 (B&D §1.6(b)).
    
    Args:
        results_df: pd.DataFrame from fit_arima_grid
        residuals_series: pd.Series, original residuals for re-fitting models
        approach_name: str, for logging
        
    Returns:
        lb_results: list of dicts {p, q, aicc, lb20_pvalue, lb_pass}
        all_pass: bool, True if all top-3 pass LB test
    """
    # Filter to causal+invertible, get top-3 by AICC
    valid_df = results_df[results_df['causal'] & results_df['invertible']]
    top3 = valid_df.head(3)
    
    if len(top3) == 0:
        print(f"  WARNING: No causal+invertible models for {approach_name}")
        return [], False
    
    lb_results = []
    all_pass = True
    
    for idx, row in top3.iterrows():
        p, q = int(row['p']), int(row['q'])
        
        try:
            # Re-fit to get residuals for LB test
            model = ARIMA(residuals_series, order=(p, 0, q), trend='n')
            result = model.fit(method='statespace')
            
            # Ljung-Box test h=20 (B&D §1.6(b))
            lb_test = acorr_ljungbox(result.resid, lags=[20], return_df=True)
            lb_pvalue = float(lb_test['lb_pvalue'].iloc[0])
            lb_pass = lb_pvalue > 0.05  # Pass criterion (D-10)
            
            lb_results.append({
                'p': p,
                'q': q,
                'aicc': float(row['aicc']),
                'lb20_pvalue': lb_pvalue,
                'lb_pass': lb_pass,
            })
            
            if not lb_pass:
                all_pass = False
        except Exception as e:
            print(f"  LB test failed for ({p},{q}): {str(e)[:50]}")
            all_pass = False
    
    return lb_results, all_pass

print("Ljung-Box test function defined.")

In [ ]:
# Run LB tests for all three approaches
print("\nRunning Ljung-Box tests on top-3 models...\n")

lb_classical, lb_classical_pass = ljung_box_test_top3(results_classical_df, residual_classical, "classical")
lb_differencing, lb_differencing_pass = ljung_box_test_top3(results_differencing_df, residual_differencing, "differencing")
lb_harmonic_gls, lb_harmonic_gls_pass = ljung_box_test_top3(results_harmonic_gls_df, residual_harmonic_gls, "harmonic_gls")

In [ ]:
# Display LB results
for approach_name, lb_results in [
    ("classical", lb_classical),
    ("differencing", lb_differencing),
    ("harmonic_gls", lb_harmonic_gls),
]:
    print(f"\n{approach_name.upper()} — Top-3 Ljung-Box Results (h=20):")
    if len(lb_results) == 0:
        print(f"  (No causal+invertible models)")
    else:
        for r in lb_results:
            status = "PASS" if r['lb_pass'] else "FAIL"
            print(f"  ARIMA({r['p']},0,{r['q']}): AICC={r['aicc']:.1f}, LB p-value={r['lb20_pvalue']:.4f} [{status}]")

## Section 7: Extension Logic and Per-Approach Winner Declaration

**Logic per D-11:** If all top-3 models fail LB test (p-value ≤ 0.05), extend grid from {0..5} to {0..6}. Re-run LB test on new top-3.

**Winner declaration per D-13:** Lowest AICC model that is causal, invertible, and passes LB (p > 0.05). If no LB-pass model found after extension, select lowest AICC causal+invertible.

In [ ]:
# Check if extension needed per approach
grid_extended_classical = not lb_classical_pass
grid_extended_differencing = not lb_differencing_pass
grid_extended_harmonic_gls = not lb_harmonic_gls_pass

print(f"Extension needed: classical={grid_extended_classical}, differencing={grid_extended_differencing}, harmonic_gls={grid_extended_harmonic_gls}")

In [ ]:
def extend_grid_and_retest(residuals_series, results_df, approach_name, max_p=6, max_q=6):
    """
    Extend grid from {0..5} to {0..6} and re-run LB test on new top-3 (D-11).
    
    Returns: extended_df, new_lb_results, all_new_pass
    """
    print(f"Extending grid for {approach_name} to p,q ∈ {{0..{max_p}}}...")
    
    # Fit only new models not already in original grid
    extended_results = []
    existing_pairs = {(int(row['p']), int(row['q'])) for _, row in results_df.iterrows()}
    
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            if p == 0 and q == 0:
                continue
            if (p, q) in existing_pairs:
                continue  # Already fitted
            
            try:
                model = ARIMA(residuals_series, order=(p, 0, q), trend='n')
                result = model.fit(method='statespace')
                
                ar_moduli = np.abs(result.arroots) if len(result.arroots) > 0 else np.array([np.inf])
                ma_moduli = np.abs(result.maroots) if len(result.maroots) > 0 else np.array([np.inf])
                causal = bool(np.all(ar_moduli > 1.001))
                invertible = bool(np.all(ma_moduli > 1.001))
                
                extended_results.append({
                    'p': p,
                    'q': q,
                    'aicc': float(result.aicc),
                    'ar_root_min': float(np.min(ar_moduli)),
                    'ma_root_min': float(np.min(ma_moduli)),
                    'causal': causal,
                    'invertible': invertible,
                    'convergence': True,
                })
            except:
                pass
    
    # Combine, remove duplicates, re-sort
    extended_df = pd.concat(
        [results_df, pd.DataFrame(extended_results)],
        ignore_index=True
    )
    extended_df = extended_df.drop_duplicates(subset=['p', 'q']).sort_values('aicc').reset_index(drop=True)
    print(f"Extended grid: {len(extended_df)} total models")
    
    # Re-test new top-3
    valid_df = extended_df[extended_df['causal'] & extended_df['invertible']]
    new_top3 = valid_df.head(3)
    
    new_lb_results = []
    new_all_pass = True
    for idx, row in new_top3.iterrows():
        p, q = int(row['p']), int(row['q'])
        try:
            model = ARIMA(residuals_series, order=(p, 0, q), trend='n')
            result = model.fit(method='statespace')
            lb_test = acorr_ljungbox(result.resid, lags=[20], return_df=True)
            lb_pvalue = float(lb_test['lb_pvalue'].iloc[0])
            lb_pass = lb_pvalue > 0.05
            
            new_lb_results.append({
                'p': p, 'q': q, 'aicc': float(row['aicc']),
                'lb20_pvalue': lb_pvalue, 'lb_pass': lb_pass
            })
            if not lb_pass:
                new_all_pass = False
        except:
            new_all_pass = False
    
    return extended_df, new_lb_results, new_all_pass

print("Extension function defined.")

In [ ]:
# Execute extension if needed
if grid_extended_classical:
    results_classical_df, lb_classical, _ = extend_grid_and_retest(residual_classical, results_classical_df, "classical")
    print(f"Classical extension: new top-3 LB results")

if grid_extended_differencing:
    results_differencing_df, lb_differencing, _ = extend_grid_and_retest(residual_differencing, results_differencing_df, "differencing")
    print(f"Differencing extension: new top-3 LB results")

if grid_extended_harmonic_gls:
    results_harmonic_gls_df, lb_harmonic_gls, _ = extend_grid_and_retest(residual_harmonic_gls, results_harmonic_gls_df, "harmonic_gls")
    print(f"Harmonic GLS extension: new top-3 LB results")

In [ ]:
def declare_winner(results_df, lb_results, approach_name):
    """
    Declare per-approach winner (D-13).
    
    Winner = lowest AICC model that is causal+invertible and passes LB (p > 0.05).
    If no LB-pass found, select lowest AICC causal+invertible.
    
    Returns: winner_dict with keys p, q, aicc, ar_root_min, ma_root_min, lb20_pvalue
    """
    # Get LB-passing models
    lb_passing = {(r['p'], r['q']) for r in lb_results if r['lb_pass']}
    
    winner_row = None
    
    if len(lb_passing) > 0:
        # Find lowest AICC among LB-passing models
        for _, row in results_df.iterrows():
            if (int(row['p']), int(row['q'])) in lb_passing:
                winner_row = row
                break
    
    if winner_row is None:
        print(f"  WARNING: No LB-passing model for {approach_name}; selecting lowest AICC causal+invertible")
        valid_df = results_df[results_df['causal'] & results_df['invertible']]
        if len(valid_df) > 0:
            winner_row = valid_df.iloc[0]
        else:
            # Ultimate fallback: any converged model
            converged_df = results_df[results_df['convergence']]
            winner_row = converged_df.iloc[0] if len(converged_df) > 0 else results_df.iloc[0]
    
    winner_dict = {
        'p': int(winner_row['p']),
        'q': int(winner_row['q']),
        'aicc': float(winner_row['aicc']),
        'ar_root_min': float(winner_row['ar_root_min']),
        'ma_root_min': float(winner_row['ma_root_min']),
        'lb20_pvalue': next((r['lb20_pvalue'] for r in lb_results if r['p']==int(winner_row['p']) and r['q']==int(winner_row['q'])), np.nan),
    }
    
    print(f"\n{approach_name.upper()} WINNER: ARIMA({winner_dict['p']},0,{winner_dict['q']}) with AICC={winner_dict['aicc']:.1f}")
    return winner_dict

print("Winner declaration function defined.")

In [ ]:
# Declare per-approach winners
winner_classical = declare_winner(results_classical_df, lb_classical, "classical")
winner_differencing = declare_winner(results_differencing_df, lb_differencing, "differencing")
winner_harmonic_gls = declare_winner(results_harmonic_gls_df, lb_harmonic_gls, "harmonic_gls")

**Note:** Phase 5 will perform full residual diagnostics (LB at h=20, 30, 40; QQ plot) and select the overall cross-approach winner after MLE fitting.

## Section 8: AICC Heatmap Visualizations

**Reference:** D-16 — 6×6 (or 7×7 if extended) AICC heatmap with color intensity proportional to model quality. Winner marked with red star.

In [ ]:
def plot_aicc_heatmap(results_df, approach_name, winner_dict, max_p=5, max_q=5):
    """
    Create AICC heatmap with winner marked (D-16).
    """
    # Create matrix (NaN for missing models)
    aicc_matrix = np.full((max_p + 1, max_q + 1), np.nan)
    
    for _, row in results_df.iterrows():
        p, q = int(row['p']), int(row['q'])
        if p <= max_p and q <= max_q:
            aicc_matrix[p, q] = row['aicc']
    
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(aicc_matrix, cmap='RdYlGn_r', aspect='auto', origin='lower')
    
    ax.set_xlabel('q (MA order)', fontsize=12)
    ax.set_ylabel('p (AR order)', fontsize=12)
    ax.set_xticks(range(max_q + 1))
    ax.set_yticks(range(max_p + 1))
    ax.set_xticklabels(range(max_q + 1))
    ax.set_yticklabels(range(max_p + 1))
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('AICC', fontsize=11)
    
    # Mark winner with red star
    winner_p, winner_q = winner_dict['p'], winner_dict['q']
    ax.plot(winner_q, winner_p, 'r*', markersize=30, markeredgecolor='darkred', markeredgewidth=1.5)
    
    ax.set_title(f'AICC Heatmap: {approach_name} — Winner ARIMA({winner_p},0,{winner_q})', fontsize=13, pad=15)
    plt.tight_layout()
    plt.savefig(config.RESULTS_DIR / f'aicc_heatmap_{approach_name}.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
    plt.show()

print("Heatmap plotting function defined.")

In [ ]:
# Generate heatmaps for all three approaches
max_p_display = 6 if grid_extended_classical else 5
plot_aicc_heatmap(results_classical_df, "classical", winner_classical, max_p=max_p_display, max_q=max_p_display)

max_p_display = 6 if grid_extended_differencing else 5
plot_aicc_heatmap(results_differencing_df, "differencing", winner_differencing, max_p=max_p_display, max_q=max_p_display)

max_p_display = 6 if grid_extended_harmonic_gls else 5
plot_aicc_heatmap(results_harmonic_gls_df, "harmonic_gls", winner_harmonic_gls, max_p=max_p_display, max_q=max_p_display)

## Section 9: Ranked AICC Tables (Top 10 per Approach)

**Reference:** D-17 — Ranked table of all converged models with AICC, root moduli, and causal/invertible flags.

In [ ]:
def display_ranked_table(results_df, approach_name):
    """
    Display top-10 models ranked by AICC (D-17).
    """
    display_df = results_df[['p', 'q', 'aicc', 'ar_root_min', 'ma_root_min', 'causal', 'invertible']].copy()
    display_df = display_df.head(10).reset_index(drop=True)
    
    # Format for readability
    display_df['p'] = display_df['p'].astype(int)
    display_df['q'] = display_df['q'].astype(int)
    display_df['aicc'] = display_df['aicc'].apply(lambda x: f"{x:.1f}" if not np.isnan(x) else "NaN")
    display_df['ar_root_min'] = display_df['ar_root_min'].apply(lambda x: f"{x:.4f}" if not np.isnan(x) else "NaN")
    display_df['ma_root_min'] = display_df['ma_root_min'].apply(lambda x: f"{x:.4f}" if not np.isnan(x) else "NaN")
    
    print(f"\n{approach_name.upper()} — Top 10 Models by AICC:")
    print(display_df.to_string(index=False))

display_ranked_table(results_classical_df, "classical")
display_ranked_table(results_differencing_df, "differencing")
display_ranked_table(results_harmonic_gls_df, "harmonic_gls")

## Section 10: Write results/model_selection.json

**Output schema per D-19:** Complete overwrite of results/model_selection.json with three approaches (classical, differencing, harmonic_gls), each containing full AICC tables, winners, and Ljung-Box results.

In [ ]:
def prepare_json_output(results_df, lb_results, winner_dict, approach_name, n_obs, grid_extended):
    """
    Prepare JSON sub-block for one approach per D-19 schema.
    """
    # Build full table with LB results
    full_table = []
    for _, row in results_df.iterrows():
        lb_pvalue = next(
            (r['lb20_pvalue'] for r in lb_results if r['p']==int(row['p']) and r['q']==int(row['q'])),
            None
        )
        lb_pass = next(
            (r['lb_pass'] for r in lb_results if r['p']==int(row['p']) and r['q']==int(row['q'])),
            None
        )
        full_table.append({
            'p': int(row['p']),
            'q': int(row['q']),
            'aicc': float(row['aicc']) if not np.isnan(row['aicc']) else None,
            'ar_root_min': float(row['ar_root_min']) if not np.isnan(row['ar_root_min']) else None,
            'ma_root_min': float(row['ma_root_min']) if not np.isnan(row['ma_root_min']) else None,
            'causal': bool(row['causal']),
            'invertible': bool(row['invertible']),
            'lb20_pvalue': lb_pvalue,
            'lb_pass': lb_pass,
        })
    
    # Top-3 summary
    top3 = [
        {
            'p': r['p'],
            'q': r['q'],
            'aicc': r['aicc'],
            'lb20_pvalue': r['lb20_pvalue'],
            'lb_pass': r['lb_pass'],
        }
        for r in lb_results
    ]
    
    return {
        'n_obs': n_obs,
        'grid': f'ARIMA(p,0,q), p,q in {{0..{"6" if grid_extended else "5"}}}',
        'grid_extended': grid_extended,
        'winner': winner_dict,
        'top3': top3,
        'full_table': full_table,
    }

print("JSON preparation function defined.")

In [ ]:
# Build complete model_selection.json
model_selection = {
    'classical': prepare_json_output(
        results_classical_df, lb_classical, winner_classical,
        'classical', len(residual_classical), grid_extended_classical
    ),
    'differencing': prepare_json_output(
        results_differencing_df, lb_differencing, winner_differencing,
        'differencing', len(residual_differencing), grid_extended_differencing
    ),
    'harmonic_gls': prepare_json_output(
        results_harmonic_gls_df, lb_harmonic_gls, winner_harmonic_gls,
        'harmonic_gls', len(residual_harmonic_gls), grid_extended_harmonic_gls
    ),
}

print("model_selection dictionary prepared.")
print(f"Keys: {list(model_selection.keys())}")
print(f"Classical winner: ARIMA({winner_classical['p']},0,{winner_classical['q']})")
print(f"Differencing winner: ARIMA({winner_differencing['p']},0,{winner_differencing['q']})")
print(f"Harmonic GLS winner: ARIMA({winner_harmonic_gls['p']},0,{winner_harmonic_gls['q']})")

In [ ]:
# Write to file
output_path = Path(config.MODEL_SELECTION_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(model_selection, f, indent=2, default=float)

print(f"\nmodel_selection.json written to {output_path}")
print(f"File size: {output_path.stat().st_size} bytes")
print(f"\n✓ Phase 4 complete.")
print(f"  Classical:    ARIMA({winner_classical['p']},0,{winner_classical['q']}) AICC={winner_classical['aicc']:.1f}")
print(f"  Differencing: ARIMA({winner_differencing['p']},0,{winner_differencing['q']}) AICC={winner_differencing['aicc']:.1f}")
print(f"  Harmonic GLS: ARIMA({winner_harmonic_gls['p']},0,{winner_harmonic_gls['q']}) AICC={winner_harmonic_gls['aicc']:.1f}")